In [1]:
# 1. ENVIRONMENT SETUP
!pip install -q ultralytics opencv-python-headless
from google.colab import drive
drive.mount('/content/drive')

import cv2
import os
from collections import deque
from ultralytics import YOLO
from tqdm.notebook import tqdm # Import tqdm for progress bars

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 27.6 MB/s eta 0:00:00
Mounted at /content/drive
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [7]:
# 2. PIPELINE CONFIGURATION
# Update these paths to match your Drive structure
MODEL_PATH = '/content/drive/MyDrive/basketball_trigger_nano.pt'
INPUT_VIDEO = '/content/drive/MyDrive/test_stream1.mp4'
OUTPUT_DIR = '/content/clips/'

In [8]:
# 3. THE GEOMETRIC HEURISTIC
def is_rim_event(ball_box, rim_box, margin=15):
    """Checks if ball intersects rim OR is directly above and aligned with it."""
    # Intersection Check
    x1 = max(ball_box[0] - margin, rim_box[0] - margin)
    y1 = max(ball_box[1] - margin, rim_box[1] - margin)
    x2 = min(ball_box[2] + margin, rim_box[2] + margin)
    y2 = min(ball_box[3] + margin, rim_box[3] + margin)
    if x1 < x2 and y1 < y2:
        return True

    # Ball Above Rim & Horizontally Aligned
    # y-axis increases from top to bottom in OpenCV
    ball_above = ball_box[3] < rim_box[1]
    horizontally_aligned = (ball_box[2] >= rim_box[0] - margin) and (ball_box[0] <= rim_box[2] + margin)

    return ball_above and horizontally_aligned

In [9]:
# 4. THE STATE MACHINE
def extract_highlights(video_path, model_path, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    model = YOLO(model_path)
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        raise ValueError("Failed to open video stream.")

    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    shooter_pre, shooter_post = int(1 * fps), int(5 * fps)
    rim_pre, rim_post = int(3 * fps), int(3 * fps)

    max_pre_frames = max(shooter_pre, rim_pre)
    ring_buffer = deque(maxlen=max_pre_frames)
    shooter_history = deque([0]*5, maxlen=5)
    rim_history = deque([0]*5, maxlen=5)

    recording = False
    frames_left = 0
    clip_idx = 0
    writer = None

    # Wrap the loop with tqdm for progress tracking
    with tqdm(total=total_frames, desc="Processing Video") as pbar:
        while True:
            ret, frame = cap.read()
            if not ret: break

            ring_buffer.append(frame)

            # Explicitly set device=0 for GPU acceleration
            results = model.predict(frame, conf=0.5, verbose=False, device=0)

            boxes = results[0].boxes.xyxy.cpu().numpy() if len(results[0].boxes) else []
            classes = results[0].boxes.cls.cpu().numpy() if len(results[0].boxes) else []

            has_shooter, has_rim_event = 0, 0
            balls, rims = [], []

            for box, cls_id in zip(boxes, classes):
                if int(cls_id) == 0: has_shooter = 1
                elif int(cls_id) == 1: balls.append(box)
                elif int(cls_id) == 2: rims.append(box)

            if balls and rims:
                for b in balls:
                    for r in rims:
                        if is_rim_event(b, r):
                            has_rim_event = 1
                            break
                    if has_rim_event: break

            shooter_history.append(has_shooter)
            rim_history.append(has_rim_event)

            shooter_trigger = sum(shooter_history) >= 3
            rim_trigger = sum(rim_history) >= 3

            # Only start a new clip if we are not already recording
            if (shooter_trigger or rim_trigger) and not recording:
                recording = True

                # Determine clip type bounds based on your rules
                if rim_trigger:
                    pre_needed = rim_pre
                    frames_left = rim_post
                else:
                    pre_needed = shooter_pre
                    frames_left = shooter_post

                clip_path = os.path.join(out_dir, f"highlight_{clip_idx:04d}.mp4")
                writer = cv2.VideoWriter(clip_path, fourcc, fps, (width, height))
                clip_idx += 1

                for buf_frame in list(ring_buffer)[-pre_needed:]:
                    writer.write(buf_frame)

                shooter_history.clear(); shooter_history.extend([0]*5)
                rim_history.clear(); rim_history.extend([0]*5)

            if recording:
                writer.write(frame)
                frames_left -= 1
                if frames_left <= 0:
                    recording = False
                    writer.release()
                    writer = None
            pbar.update(1)

    if writer: writer.release()
    cap.release()
    print(f"Extraction complete. {clip_idx} highlights saved to {out_dir}")

# 5. EXECUTE
extract_highlights(INPUT_VIDEO, MODEL_PATH, OUTPUT_DIR)

Processing Video:   0%|          | 0/7344 [00:00<?, ?it/s]

Extraction complete. 43 highlights saved to /content/clips/


In [10]:
import shutil

source_dir = '/content/clips/'
dest_dir = '/content/drive/MyDrive/clips_test1/'

# Copy the clips folder to your Google Drive
shutil.copytree(source_dir, dest_dir, dirs_exist_ok=True)
print(f"Clips successfully saved to {dest_dir}")

Clips successfully saved to /content/drive/MyDrive/clips_test1/
